<img src="images/Title.png" width="1000" height="1000">

## What is a Knowledge Graph?

A **Knowledge Graph** is a structured representation of real-world entities and the relationships between them.

It doesn't just tell us:

> "John works on Project A."

It represents the knowledge as:

```text
John
  │
  └── WORKS_ON ──► Project A
                         │
                         └── USES ──► Azure OpenAI

                     
```

### Simple Analogy

<img src="images/grapgrag-analogy.png" width="1000" height="1000">

### Real Example


<img src="images/knowledgegraph-example.png" width="1000" height="1000">



## What is a Graph Database?

A **Graph Database** is a database designed to store and query highly connected data.

Instead of storing data primarily as rows and tables, it represents data using:

- **Nodes** → Entities
- **Relationships** → Connections between entities
- **Properties** → Information about nodes and relationships

### Simple Example

Employee → **WORKS_ON** → Project

Project → **USES** → Technology

Employee → **BELONGS_TO** → Department

The biggest advantage is that relationships are stored natively, making it easy to traverse connected data



<img src="images/neo4j.webp" width="300" height="300">

**Neo4j** is a **native graph database** that uses the property-graph model.

It provides:

- Nodes
- Relationships
- Properties
- Labels
- Cypher query language
- Graph visualization and developer tools

For our project, Neo4j will act as the **storage and query layer for our Knowledge Graph**.

```text
                 Knowledge Graph
                       │
                       ▼
              ┌─────────────────┐
              │      Neo4j      │
              │                 │
              │ Nodes           │
              │ Relationships   │
              │ Properties      │
              └─────────────────┘

---



| Graph Database                  | Knowledge Graph                        |
| ------------------------------- | -------------------------------------- |
| **Technology / Storage**        | **Knowledge / Data Model**             |
| Stores nodes and relationships  | Organizes entities and their meaning   |
| Provides querying and traversal | Represents real-world knowledge        |
| Example: **Neo4j**              | Example: Employee → WORKS_ON → Project |


<img src="images/GraphRAGFlow.png" width="1000" height="1400">

### 1. Installations

In [ ]:
! uv pip install jupyter ipykernel

In [ ]:
# python -m ipykernel install --user --name=graphrag --display-name "Python (GraphRAG)"

In [ ]:
!uv pip install langchain_community pypdf langchain langchain_openai langchain_experimental langchain_neo4j


### 2. Data Load

In [ ]:
# ============================================================
# Configure PDF Directory
# ============================================================


from pathlib import Path

DATA_DIR = Path("data")
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f" - {pdf.name}")

In [ ]:
# ============================================================
# Load PDF Documents
# ============================================================


from langchain_community.document_loaders import PyPDFLoader

documents = []

for pdf_path in pdf_files:
    print(f"Loading: {pdf_path.name}")

    loader = PyPDFLoader(str(pdf_path))
    documents.extend(loader.load())

print("\n" + "=" * 60)
print(f"Total PDF Files : {len(pdf_files)}")
print(f"Total Pages     : {len(documents)}")
print("=" * 60)

In [ ]:
# ============================================================
# Preview Loaded Documents
# ============================================================

print(f"Source : {documents[0].metadata['source']}")
print(f"Page   : {documents[0].metadata['page']}")
print("-" * 60)
print(documents[0].page_content[:1000])

### 3. Document Chunking

In [ ]:
# ============================================================
# Split Documents into Chunks
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

split_documents = text_splitter.split_documents(documents)

print("=" * 60)
print(f"Original Documents : {len(documents)}")
print(f"Total Chunks       : {len(split_documents)}")
print("=" * 60)

### 4. Define the LLM

In [ ]:
# ============================================================
# Azure OpenAI Configuration
# ============================================================

import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI

load_dotenv()
required = ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_DEPLOYMENT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise ValueError(f"Set {', '.join(missing)} in your local .env file; see .env.example.")

llm = AzureChatOpenAI(
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview"),
    temperature=0,
)

print("Azure OpenAI model configured.")

### 5. Graph Building

In [ ]:
split_documents

In [ ]:
# ============================================================
# Convert Documents into Graph Documents
# ============================================================

from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(llm=llm)

graph_documents = llm_transformer.convert_to_graph_documents(
    split_documents[:20]
)

print("=" * 60)
print(f"Graph Documents Created : {len(graph_documents)}")
print("=" * 60)

In [ ]:
# ============================================================
# Preview Extracted Graph
# ============================================================

graph = graph_documents[0]

print("=" * 60)
print("NODES")
print("=" * 60)

for node in graph.nodes:
    print(node)

print("\n" + "=" * 60)
print("RELATIONSHIPS")
print("=" * 60)

for relationship in graph.relationships:
    print(relationship)

In [ ]:
# ============================================================
# Preview Multiple Graph Documents
# ============================================================

for index, graph in enumerate(graph_documents[:3], start=1):

    print("\n" + "=" * 70)
    print(f"Graph Document {index}")
    print("=" * 70)

    print(f"Nodes         : {len(graph.nodes)}")
    print(f"Relationships : {len(graph.relationships)}")

### 6. Setup the Neo4j Using Docker

In [ ]:
# docker pull neo4j:latest  #docker pull neo4j:5.26
# docker run -d --name neo4j-graphrag -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/GraphRAG@123 -e NEO4J_PLUGINS='["apoc"]' -e NEO4J_dbms_security_procedures_unrestricted=apoc.* -e NEO4J_dbms_security_procedures_allowlist=apoc.* -v neo4j_data:/data neo4j:latest
# docker run -d --name neo4j-graphrag -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/GraphRAG@123 -v neo4j_data:/data neo4j:5.26
'''
MATCH (n)
RETURN count(n);

MATCH ()-[r]->()
RETURN count(r);
'''

In [ ]:
# ============================================================
# Connect to Neo4j
# ============================================================

from neo4j import GraphDatabase

import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD")
if not PASSWORD:
    raise ValueError("Set NEO4J_PASSWORD in your local .env file; see .env.example.")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print(f"Connected to Neo4j at {URI}")

### Graph DB Ingestion

In [ ]:
from neo4j import GraphDatabase


def sanitize(text: str) -> str:
    return (
        text.upper()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(".", "_")
    )


def store_graph_documents(driver, graph_documents):
    with driver.session() as session:
        for graph_document in graph_documents:

            # --------------------------------------------------
            # Store Nodes
            # --------------------------------------------------

            for node in graph_document.nodes:
                label = sanitize(node.type)
                properties = {"id": node.id, "name": node.id}

                if hasattr(node, "properties"):
                    properties.update(node.properties)

                query = f"""
                        MERGE (n:Entity:{label} {{id: $id}})
                        SET n += $properties
                        """
                session.run(query, id=node.id, properties=properties)

            # --------------------------------------------------
            # Store Relationships
            # --------------------------------------------------

            for relationship in graph_document.relationships:
                source_label = sanitize(relationship.source.type)
                target_label = sanitize(relationship.target.type)
                relationship_type = sanitize(relationship.type)
                properties = {}

                if hasattr(relationship, "properties"):
                    properties.update(relationship.properties)

                query = (
                    f"MATCH (a:{source_label} {{id: $source}}) "
                    f"MATCH (b:{target_label} {{id: $target}}) "
                    f"MERGE (a)-[r:{relationship_type}]->(b) "
                    f"SET r += $properties"
                )

                session.run(
                    query,
                    source=relationship.source.id,
                    target=relationship.target.id,
                    properties=properties,
                )

    print("✅ Graph stored successfully.")


In [ ]:
# ============================================================
# Store Graph into Neo4j
# ============================================================

store_graph_documents(
    driver=driver,
    graph_documents=graph_documents
)

print("✅ Graph successfully ingested into Neo4j.")

### Testing Commands


 1. List All Node Labels

```cypher
CALL db.labels();
```
2. List All Relationship Types
```cypher
CALL db.relationshipTypes();
```

3. View Sample Nodes
```cypher
MATCH (n)
RETURN n
LIMIT 10;
```

4. View Sample Relationships
```cypher
MATCH (a)-[r]->(b)
RETURN a, r, b
LIMIT 20;
```

5. View Sample Relationships
```cypher
MATCH (n)-[r]-(m)
WHERE n.name CONTAINS "Azure"
RETURN n, r, m;
```

## Graph DB Extraction

In [ ]:
# ============================================================
# User Question
# ============================================================

question = "Who is working on the Employee Directory project?"
question = "Tell me about Cyber Security"

In [ ]:
# ============================================================
# Retrieve Matching Graph Context
# ============================================================

from neo4j import GraphDatabase

query = """
MATCH (n)-[r]-(m)

WHERE toLower($question) CONTAINS toLower(n.name)
   OR toLower(n.name) CONTAINS toLower($question)

RETURN
    labels(n) AS source_type,
    n.name AS source,
    type(r) AS relationship,
    labels(m) AS target_type,
    m.name AS target

LIMIT 50
"""

with driver.session() as session:

    result = session.run(
        query,
        question=question
    )

    graph_context = []

    for record in result:

        graph_context.append(
            f"{record['source']} "
            f"-[{record['relationship']}]-> "
            f"{record['target']}"
        )

print("\n".join(graph_context))

In [ ]:
# ============================================================
# Generate Final Answer
# ============================================================

context = "\n".join(graph_context)

prompt = f"""
You are a helpful assistant.

Answer ONLY using the graph context.

Graph Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(response.content)

# 🔹 Case 1: LLM Entity Extraction + Direct Entity Matching


Instead of sending the complete user question to Neo4j, we first use the LLM
to extract the important entities and then directly match those entities
against the entity names stored in Neo4j.

<img src="images/case-1-exactmatch.png" width="1200" height="1400">

**Entity Extraction**

In [ ]:
# ============================================================
# User Question
# ============================================================

question = "Who is working on the Employee Directory project?"
question = "Tell me about Cyber Security"

In [ ]:
# ============================================================
# Extract Entities from User Question
# ============================================================

from pydantic import BaseModel, Field
from typing import List


class EntityExtraction(BaseModel):
    entities: List[str] = Field(
        description="List of important entities mentioned in the question."
    )


entity_llm = llm.with_structured_output(EntityExtraction)

question = "Tell me about Cyber Security"

entity_result = entity_llm.invoke(f"""
Extract all important entities from the following question.

Question:
{question}
""")

print("=" * 60)
print("Extracted Entities")
print("=" * 60)

print(entity_result.entities)

In [ ]:
entity_result

**Retrieval**

In [ ]:
# ============================================================
# Retrieve Relevant Graph Context
# ============================================================
cypher_query = """
MATCH path = (n)-[*1..2]-(m)

WHERE toLower(n.name) = toLower($entity)

UNWIND relationships(path) AS r
UNWIND nodes(path) AS node

RETURN DISTINCT
    startNode(r).name AS source,
    type(r) AS relationship,
    endNode(r).name AS target
LIMIT 100
"""

graph_context = []

with driver.session() as session:

    for entity in ['Cyber Secuuriity']:

        result = session.run(
            cypher_query,
            entity=entity
        )

        for record in result:

            graph_context.append(
                f"{record['source']} "
                f"-[{record['relationship']}]-> "
                f"{record['target']}"
            )

print("=" * 60)
print("Retrieved Graph Context")
print("=" * 60)

print("\n".join(graph_context))

**Generation**

In [ ]:
# ============================================================
# Generate Final Answer
# ============================================================

context = "\n".join(graph_context)

prompt = f"""
You are an AI assistant.

Answer ONLY using the graph context.

If the answer is not available, say so.

==========================
Graph Context
==========================

{context}

==========================
Question
==========================

{question}
"""

response = llm.invoke(prompt)

print("=" * 60)
print("Final Answer")
print("=" * 60)

print(response.content)

# 🔹 Case 2: LLM Entity Extraction + Partial Entity Matching

We keep the LLM-based entity extraction from Case 1, but instead of requiring
an exact entity-name match, we use **partial matching** to find relevant
entities in Neo4j.

<img src="images/case-2-partialmatching.png" width="1200" height="1400">

In [ ]:
# ============================================================
# Retrieve Relevant Graph Context from Neo4j
# ============================================================

cypher_query = """
MATCH (n)-[r]-(m)

WHERE ANY(keyword IN $keywords
          WHERE toLower(n.name) CONTAINS keyword)

RETURN DISTINCT
    labels(n) AS source_label,
    n.name AS source,
    type(r) AS relationship,
    labels(m) AS target_label,
    m.name AS target

LIMIT 100
"""

graph_context = []

with driver.session() as session:

    for entity in entity_result.entities:

        # Split extracted entity into keywords
        keywords = [
            word.strip()
            for word in entity.lower().split()
            if len(word.strip()) > 2
        ]

        print(f"\nSearching for Entity : {entity}")
        print(f"Keywords             : {keywords}")

        result = session.run(
            cypher_query,
            keywords=keywords
        )

        records = list(result)

        print(f"Matches Found        : {len(records)}")

        for record in records:

            graph_context.append(
                f"{record['source']} "
                f"-[{record['relationship']}]-> "
                f"{record['target']}"
            )

print("\n" + "=" * 60)
print("Retrieved Graph Context")
print("=" * 60)

print("\n".join(graph_context))

In [ ]:
question

In [ ]:
# ============================================================
# Generate Final Answer
# ============================================================

context = "\n".join(graph_context)

prompt = f"""
You are an AI assistant.

Answer ONLY using the graph context.

If the answer is not available, say so.

==========================
Graph Context
==========================

{context}

==========================
Question
==========================

{question}
"""

response = llm.invoke(prompt)

print("=" * 60)
print("Final Answer")
print("=" * 60)

print(response.content)

| Approach                      | Entity retrieval                                   | Main limitation                                          |
| ----------------------------- | -------------------------------------------------- | -------------------------------------------------------- |
| **1. Direct matching**        | Question → `Exact Match`                              | Very dependent on wording                                |
| **2. LLM + partial matching** | LLM extracts entity → `CONTAINS`                   | Better, but still keyword-based                          |
| **3. LLM + Full-Text Search** | LLM extracts entity → Neo4j Full-Text/Lucene index | Faster and scalable for large graphs + relevance scoring |


# 🔹 Case 3: LLM Entity Extraction + Neo4j Full-Text Search


In this approach, we first use the LLM to extract the important entities from
the user's question. Instead of using exact or basic partial string matching,
we use **Neo4j Full-Text Search** to find the most relevant entities from the
graph.
<img src="images/case-3-fulltextsearch.png" width="1200" height="1400">

In [ ]:
# ============================================================
# Create Full-Text Index (If Not Exists)
# ============================================================

CHECK_INDEX_QUERY = """
SHOW INDEXES
YIELD name
WHERE name = 'entity_index'
RETURN count(*) AS count
"""

CREATE_INDEX_QUERY = """
CREATE FULLTEXT INDEX entity_index
FOR (n:Entity)
ON EACH [n.name]
"""

with driver.session() as session:

    result = session.run(CHECK_INDEX_QUERY).single()

    if result["count"] == 0:

        session.run(CREATE_INDEX_QUERY)

        print("✅ Full-Text Index created successfully.")

    else:

        print("ℹ️ Full-Text Index already exists.")

In [ ]:
# ============================================================
# User Question
# ============================================================

question = "Tell me about Cyber Security"

# ============================================================
# Extract Entities
# ============================================================

from pydantic import BaseModel, Field
from typing import List


class EntityExtraction(BaseModel):
    entities: List[str] = Field(
        description="List of entities mentioned in the user question."
    )


entity_llm = llm.with_structured_output(EntityExtraction)

entity_result = entity_llm.invoke(f"""
Extract all important entities from the question.

Question:
{question}
""")

print("=" * 60)
print("Extracted Entities")
print("=" * 60)

print(entity_result.entities)


# ============================================================
# Full-Text Search
# ============================================================

FULLTEXT_QUERY = """
CALL db.index.fulltext.queryNodes(
    'entity_index',
    $entity
)
YIELD node, score

RETURN
    node.name AS name,
    labels(node) AS labels,
    score

ORDER BY score DESC
LIMIT 5
"""

matched_entities = []

with driver.session() as session:

    for entity in entity_result.entities:

        print(f"\nSearching : {entity}")

        result = session.run(
            FULLTEXT_QUERY,
            entity=entity
        )

        records = list(result)

        if not records:
            print("No match found.")
            continue

        for record in records:

            print(
                f"{record['name']} "
                f"(Score : {record['score']:.2f})"
            )

            matched_entities.append(record["name"])

In [ ]:
# ============================================================
# Retrieve Graph Context for Matched Entities
# ============================================================

GRAPH_QUERY = """
MATCH (n)-[r]-(m)
WHERE n.name IN $entities

RETURN DISTINCT
    n.name AS source,
    type(r) AS relationship,
    m.name AS target
LIMIT 100
"""

graph_context = []

with driver.session() as session:

    result = session.run(
        GRAPH_QUERY,
        entities=matched_entities
    )

    for record in result:

        graph_context.append(
            f"{record['source']} "
            f"-[{record['relationship']}]-> "
            f"{record['target']}"
        )

print("=" * 60)
print("Retrieved Graph Context")
print("=" * 60)

for item in graph_context:
    print(item)

In [ ]:
# ============================================================
# Generate Final Answer
# ============================================================

context = "\n".join(graph_context)

prompt = f"""
You are a GraphRAG assistant.

Answer the user's question using only the information
provided in the graph context.

If the graph context does not contain enough information,
say that the information is not available in the knowledge graph.

-------------------- Graph Context --------------------

{context}

---------------------- Question ------------------------

{question}

---------------------------------------------------------

Provide a concise and factual answer.
"""

response = llm.invoke(prompt)

print("=" * 60)
print("Final Answer")
print("=" * 60)

print(response.content)

# Next Advancements for GraphRAG

- **Hybrid GraphRAG** — Combine vector search with graph traversal.
- **Neo4j Vector Index** — Use embeddings for semantic retrieval.
- **Better Entity Resolution** — Handle synonyms, abbreviations, and ambiguous entities.
- **Multi-hop Reasoning** — Traverse multiple levels of relationships.
- **Chunk + Graph Linking** — Connect source document chunks to graph entities.
- **Relationship-aware Retrieval** — Retrieve the most relevant graph paths.
- **Reranking** — Rerank retrieved nodes, paths, and chunks before generation.
- **Source Citations** — Return the original PDF/page/chunk supporting each answer.
- **Hybrid Search** — Combine keyword, vector, and graph-based retrieval.
- **Evaluation** — Compare Traditional RAG vs GraphRAG vs Hybrid GraphRAG.